# Metaflora Incubus v1 — Kaggle recovery/export

Resumes the completed `incubus-v1-refine-001` artifact pipeline only. It checks RAM and disk, reuses verified completed stages, exports Q5_K_M and writes SHA-256 evidence. Training is never started.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

run_id = "incubus-v1-refine-001"
quantization = "Q5_K_M"
root = Path("/kaggle/working")
if not root.is_dir():
    raise RuntimeError("this recovery notebook requires a Kaggle runtime")
repository = root / "metaflora-incubus"
secrets = UserSecretsClient()
revision = secrets.get_secret("INCUBUS_CODE_REVISION")
if not isinstance(revision, str) or re.fullmatch(r"[0-9a-f]{40}", revision) is None:
    raise RuntimeError("INCUBUS_CODE_REVISION must be an exact lowercase commit SHA")
shutil.rmtree(repository, ignore_errors=True)
subprocess.run(["git", "init", str(repository)], check=True)
subprocess.run(["git", "-C", str(repository), "remote", "add", "origin", "https://github.com/metaflora-app/metaflora-incubus.git"], check=True)
subprocess.run(["git", "-C", str(repository), "fetch", "--depth", "1", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repository), "checkout", "--detach", "FETCH_HEAD"], check=True)
checked_out = subprocess.run(["git", "-C", str(repository), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip().lower()
if checked_out != revision:
    raise RuntimeError("trusted code revision verification failed")
recovery_script = repository / "scripts/kaggle_recover_export.py"
if not recovery_script.is_file():
    raise RuntimeError("pinned recovery script is missing")
subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(repository / "requirements/recovery-linux.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repository)], check=True)
sys.path.insert(0, str(repository / "src"))
from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap
encoded_bootstrap = secrets.get_secret("INCUBUS_BOOTSTRAP")
if not isinstance(encoded_bootstrap, str) or not encoded_bootstrap.strip():
    raise RuntimeError("INCUBUS_BOOTSTRAP is empty")
bootstrap_payload = decrypt_cloud_bootstrap((repository / "configs/cloud/bootstrap-v1.enc").read_bytes(), encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
os.environ.setdefault("INCUBUS_CHECKPOINT_LOCATION", "metaflora/incubus-checkpoints")
os.environ.setdefault("INCUBUS_CHECKPOINT_BRANCH", "incubus-training-v1")

base_path = os.environ.get("INCUBUS_BASE_MODEL_PATH")
adapter_path = os.environ.get("INCUBUS_ADAPTER_PATH")
if not base_path or not adapter_path:
    from huggingface_hub import snapshot_download
    required = ("INCUBUS_SOURCE_REPO", "INCUBUS_SOURCE_REVISION", "INCUBUS_CHECKPOINT_LOCATION", "INCUBUS_CHECKPOINT_BRANCH")
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError(f"missing recovery environment: {', '.join(missing)}")
    base_path = snapshot_download(repo_id=os.environ["INCUBUS_SOURCE_REPO"], revision=os.environ["INCUBUS_SOURCE_REVISION"], local_dir=root / "incubus-recovery-inputs/source")
    checkpoint_patterns = [f"runs/{run_id}/incubus-checkpoint-manifest.json", f"runs/{run_id}/final-adapter/**"]
    checkpoint_root = snapshot_download(repo_id=os.environ["INCUBUS_CHECKPOINT_LOCATION"], revision=os.environ["INCUBUS_CHECKPOINT_BRANCH"], allow_patterns=checkpoint_patterns, local_dir=root / "incubus-recovery-inputs/checkpoint")
    restored_run = Path(checkpoint_root) / "runs" / run_id
    from metaflora_incubus.cloud_training_runtime import _authenticated_recovery_binding
    _authenticated_recovery_binding(restored_run, environment=os.environ, checkpoint_key=os.environ["INCUBUS_CHECKPOINT_HMAC_KEY"], run_id=run_id)
    adapter_path = str(restored_run / "final-adapter")

llama_cpp = root / "llama.cpp-incubus-recovery"
llama_revision = json.loads((repository / "configs/cloud/free-gpu-v1.json").read_text())["llama_cpp_revision"]
if not (llama_cpp / "build/bin/llama-quantize").is_file():
    shutil.rmtree(llama_cpp, ignore_errors=True)
    subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/ggerganov/llama.cpp.git", str(llama_cpp)], check=True)
    subprocess.run(["git", "-C", str(llama_cpp), "checkout", "--detach", llama_revision], check=True)
    subprocess.run(["cmake", "-S", str(llama_cpp), "-B", str(llama_cpp / "build"), "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF"], check=True)
    subprocess.run(["cmake", "--build", str(llama_cpp / "build"), "--config", "Release", "-j", "2", "--target", "llama-quantize"], check=True)

command = [sys.executable, str(recovery_script), "--run-id", run_id, "--base", str(base_path), "--adapter", str(adapter_path), "--workspace", str(root / "incubus-v1-refine-001-export"), "--merge-script", str(repository / "scripts/merge_adapter_for_export.py"), "--convert-script", str(llama_cpp / "convert_hf_to_gguf.py"), "--quantize-binary", str(llama_cpp / "build/bin/llama-quantize")]
print(f"resuming {run_id} export as {quantization}")
subprocess.run(command, check=True, cwd=repository)
